In [ ]:

# ============================================================
# TALTECH AI PIPELINE
# NOTEBOOK 05 — PREPARE SYNCHRONIZED AI-READY DATASET
# Portable Windows/Linux version
#
# INPUT:
# outputs/04_sparql/03_ml_extraction_long_traceable.csv
#
# PROTOCOL:
# - fixed study interval: 2025-02-07 12:15 UTC -> 2025-06-27 17:45 UTC
# - 15-minute grid
# - aggregate collisions within the same rounded 15-min slot
# - interpolate ONLY internal gaps <= 4 slots (<= 1 hour)
# - preserve long gaps as missing
# - create a synchronized complete-case dataset for downstream ML
# ============================================================

from pathlib import Path
import time
import numpy as np
import pandas as pd

# ============================================================
# 1. PORTABLE PATHS
# ============================================================

PROJECT_DIR = Path.cwd().resolve()
OUTPUT_DIR = PROJECT_DIR / "outputs"
SPARQL_DIR = OUTPUT_DIR / "04_sparql"
OUT = OUTPUT_DIR / "05_ml_dataset"

INPUT_FILE = SPARQL_DIR / "03_ml_extraction_long_traceable.csv"

OUT.mkdir(parents=True, exist_ok=True)

print("====================================")
print("NOTEBOOK 05 — AI-READY DATASET")
print("====================================")
print("Project directory :", PROJECT_DIR)
print("Input file        :", INPUT_FILE)
print("Output directory  :", OUT)

assert INPUT_FILE.exists(), (
    "\nSPARQL ML extraction file not found:\n"
    f"{INPUT_FILE}\n\n"
    "Run Notebook 04 first."
)

# ============================================================
# 2. FIXED EXPERIMENTAL PROTOCOL
# ============================================================

STUDY_START = pd.Timestamp("2025-02-07 12:15:00", tz="UTC")
STUDY_END   = pd.Timestamp("2025-06-27 17:45:00", tz="UTC")

FREQ = "15min"
SHORT_GAP_LIMIT_SLOTS = 4

EXPECTED_DEVICES = 7
EXPECTED_VARIABLES = {
    "temperature",
    "relative_humidity",
    "co2"
}

EXPECTED_GRID_SLOTS = len(
    pd.date_range(
        STUDY_START,
        STUDY_END,
        freq=FREQ,
        tz="UTC"
    )
)

print("\n====================================")
print("EXPERIMENTAL PROTOCOL")
print("====================================")
print("Study start          :", STUDY_START)
print("Study end            :", STUDY_END)
print("Temporal resolution  :", FREQ)
print("Short-gap limit      :", SHORT_GAP_LIMIT_SLOTS, "slots (= 1 hour)")
print("Theoretical slots    :", EXPECTED_GRID_SLOTS)

# ============================================================
# 3. LOAD SPARQL-DERIVED DATA
# ============================================================

start = time.perf_counter()

df = pd.read_csv(
    INPUT_FILE,
    usecols=[
        "uuid",
        "timestamp",
        "variable",
        "value",
        "unit",
        "point",
        "observation"
    ]
)

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    utc=True,
    errors="coerce"
)

df["value"] = pd.to_numeric(
    df["value"],
    errors="coerce"
)

load_seconds = time.perf_counter() - start

print("\n====================================")
print("INPUT SUMMARY")
print("====================================")
print("Rows loaded          :", len(df))
print("Devices              :", df["uuid"].nunique())
print("Variables            :", sorted(df["variable"].dropna().unique()))
print(f"Load time            : {load_seconds:.2f} s")

assert df["uuid"].nunique() == EXPECTED_DEVICES
assert set(df["variable"].dropna().unique()) == EXPECTED_VARIABLES

invalid_ts = int(df["timestamp"].isna().sum())
invalid_values = int(df["value"].isna().sum())

print("Invalid timestamps   :", invalid_ts)
print("Invalid numeric value:", invalid_values)

assert invalid_ts == 0, "Invalid timestamps detected."
assert invalid_values == 0, "Invalid numeric values detected."

# ============================================================
# 4. FILTER TO FIXED STUDY INTERVAL
# ============================================================

study = df.loc[
    df["timestamp"].between(
        STUDY_START,
        STUDY_END,
        inclusive="both"
    )
].copy()

print("\n====================================")
print("FIXED STUDY INTERVAL")
print("====================================")
print("Rows retained :", len(study))
print("Min timestamp :", study["timestamp"].min())
print("Max timestamp :", study["timestamp"].max())

# ============================================================
# 5. PROJECT OBSERVATIONS TO 15-MINUTE GRID
# ============================================================
# TalTech timestamps are projected to their nearest 15-minute slot.
# Multiple source observations landing in the same
# (device, variable, slot) are averaged.

study["slot"] = study["timestamp"].dt.round(FREQ)

# Keep only projected slots inside the fixed study interval.
study = study.loc[
    study["slot"].between(
        STUDY_START,
        STUDY_END,
        inclusive="both"
    )
].copy()

collision_counts = (
    study.groupby(
        ["uuid", "variable", "slot"],
        observed=True
    )
    .size()
    .rename("source_observations")
    .reset_index()
)

collision_summary = (
    collision_counts["source_observations"]
    .value_counts()
    .sort_index()
    .rename_axis("source_observations_per_slot")
    .reset_index(name="slot_count")
)

collision_summary.to_csv(
    OUT / "01_collision_summary.csv",
    index=False
)

projected = (
    study.groupby(
        ["uuid", "variable", "slot"],
        observed=True,
        as_index=False
    )
    .agg(
        value=("value", "mean"),
        source_observations=("value", "size")
    )
)

print("\n====================================")
print("15-MIN PROJECTION")
print("====================================")
print("Raw rows in interval       :", len(study))
print("Unique projected values    :", len(projected))
print(
    "Slots with collisions      :",
    int((projected["source_observations"] > 1).sum())
)

# ============================================================
# 6. CREATE COMPLETE 15-MIN GRID FOR 7 × 3 SERIES
# ============================================================

devices = sorted(projected["uuid"].unique())
variables = sorted(EXPECTED_VARIABLES)

full_grid = pd.date_range(
    STUDY_START,
    STUDY_END,
    freq=FREQ,
    tz="UTC"
)

assert len(full_grid) == EXPECTED_GRID_SLOTS
assert len(devices) == EXPECTED_DEVICES

series_frames = []
audit_rows = []

for uuid in devices:
    for variable in variables:

        s = (
            projected.loc[
                (projected["uuid"] == uuid)
                &
                (projected["variable"] == variable),
                ["slot", "value"]
            ]
            .drop_duplicates("slot")
            .set_index("slot")["value"]
            .sort_index()
            .reindex(full_grid)
        )

        available_before = int(s.notna().sum())
        missing_before = int(s.isna().sum())

        # ----------------------------------------------------
        # Interpolate only internal gaps of <= 4 consecutive
        # missing 15-min slots. Long gaps remain NaN.
        #
        # limit_area='inside' prevents extrapolation at edges.
        # ----------------------------------------------------

        s_interp = s.interpolate(
            method="time",
            limit=SHORT_GAP_LIMIT_SLOTS,
            limit_area="inside"
        )

        # Pandas `limit` alone can partially fill a long gap.
        # We must explicitly restore every original missing run
        # whose total length is > SHORT_GAP_LIMIT_SLOTS.
        missing_mask = s.isna()

        run_id = (
            missing_mask
            != missing_mask.shift(fill_value=False)
        ).cumsum()

        missing_runs = (
            pd.DataFrame({
                "missing": missing_mask,
                "run_id": run_id
            })
            .loc[missing_mask]
            .groupby("run_id")
            .size()
        )

        long_run_ids = set(
            missing_runs[
                missing_runs > SHORT_GAP_LIMIT_SLOTS
            ].index
        )

        if long_run_ids:
            restore_mask = (
                missing_mask
                &
                run_id.isin(long_run_ids)
            )
            s_interp.loc[restore_mask] = np.nan

        available_after = int(s_interp.notna().sum())
        interpolated_values = int(
            (s.isna() & s_interp.notna()).sum()
        )
        missing_after = int(s_interp.isna().sum())

        max_gap_slots = (
            int(missing_runs.max())
            if len(missing_runs)
            else 0
        )

        audit_rows.append({
            "uuid": uuid,
            "variable": variable,
            "theoretical_slots": len(full_grid),
            "available_before_interpolation": available_before,
            "missing_before_interpolation": missing_before,
            "interpolated_short_gap_values": interpolated_values,
            "available_after_interpolation": available_after,
            "missing_after_interpolation": missing_after,
            "completeness_before_pct":
                100 * available_before / len(full_grid),
            "completeness_after_pct":
                100 * available_after / len(full_grid),
            "max_original_gap_slots": max_gap_slots,
            "max_original_gap_hours":
                max_gap_slots * 0.25
        })

        tmp = pd.DataFrame({
            "timestamp": full_grid,
            "uuid": uuid,
            "variable": variable,
            "value_original_grid": s.values,
            "value": s_interp.values
        })

        tmp["was_interpolated"] = (
            tmp["value_original_grid"].isna()
            &
            tmp["value"].notna()
        )

        series_frames.append(tmp)

audit = pd.DataFrame(audit_rows)
long_sync = pd.concat(
    series_frames,
    ignore_index=True
)

audit.to_csv(
    OUT / "02_series_temporal_quality_audit.csv",
    index=False
)

print("\n====================================")
print("TEMPORAL QUALITY")
print("====================================")
print(
    audit[
        [
            "uuid",
            "variable",
            "theoretical_slots",
            "available_before_interpolation",
            "interpolated_short_gap_values",
            "available_after_interpolation",
            "missing_after_interpolation",
            "max_original_gap_hours"
        ]
    ].to_string(index=False)
)

print(
    "\nTotal theoretical positions :",
    int(audit["theoretical_slots"].sum())
)
print(
    "Available before           :",
    int(audit["available_before_interpolation"].sum())
)
print(
    "Short-gap values filled    :",
    int(audit["interpolated_short_gap_values"].sum())
)
print(
    "Available after            :",
    int(audit["available_after_interpolation"].sum())
)

# ============================================================
# 7. CREATE SYNCHRONIZED WIDE DATASET
# ============================================================

wide = (
    long_sync
    .pivot(
        index=["uuid", "timestamp"],
        columns="variable",
        values="value"
    )
    .reset_index()
)

wide.columns.name = None

# Required order
wide = wide[
    [
        "uuid",
        "timestamp",
        "temperature",
        "relative_humidity",
        "co2"
    ]
].sort_values(
    ["uuid", "timestamp"]
).reset_index(drop=True)

wide["complete_trhco2"] = (
    wide[
        [
            "temperature",
            "relative_humidity",
            "co2"
        ]
    ]
    .notna()
    .all(axis=1)
)

complete = (
    wide.loc[
        wide["complete_trhco2"]
    ]
    .drop(columns="complete_trhco2")
    .copy()
)

print("\n====================================")
print("SYNCHRONIZED DATASET")
print("====================================")
print("All device-time rows :", len(wide))
print("Complete T/RH/CO2    :", len(complete))
print(
    "Completeness         :",
    f"{100 * len(complete) / len(wide):.2f}%"
)

# ============================================================
# 8. DETECT LONG DISCONTINUITIES IN COMPLETE DATA
# ============================================================

segment_frames = []
segment_summary_rows = []

for uuid, part in complete.groupby("uuid", sort=True):

    part = part.sort_values("timestamp").copy()

    delta = part["timestamp"].diff()

    # A new segment starts whenever consecutive retained rows
    # are separated by more than 15 minutes.
    part["segment_id"] = (
        delta.gt(pd.Timedelta(minutes=15))
        .fillna(False)
        .cumsum()
        + 1
    )

    segment_frames.append(part)

    for segment_id, seg in part.groupby("segment_id"):

        segment_summary_rows.append({
            "uuid": uuid,
            "segment_id": int(segment_id),
            "start": seg["timestamp"].min(),
            "end": seg["timestamp"].max(),
            "rows": len(seg),
            "duration_hours":
                (
                    (
                        seg["timestamp"].max()
                        -
                        seg["timestamp"].min()
                    ).total_seconds()
                    / 3600
                )
        })

complete_segmented = pd.concat(
    segment_frames,
    ignore_index=True
)

segments = pd.DataFrame(
    segment_summary_rows
)

segments.to_csv(
    OUT / "03_contiguous_segments.csv",
    index=False
)

# ============================================================
# 9. SAVE DATASETS
# ============================================================

long_sync.to_csv(
    OUT / "04_synchronized_long_with_gap_flags.csv",
    index=False
)

wide.to_csv(
    OUT / "05_synchronized_wide_all_slots.csv",
    index=False
)

complete_segmented.to_csv(
    OUT / "06_ai_ready_complete_case.csv",
    index=False
)

# ============================================================
# 10. DATASET-LEVEL AUDIT
# ============================================================

dataset_audit = pd.DataFrame([
    {
        "metric": "source_sparql_rows",
        "value": len(df)
    },
    {
        "metric": "study_interval_rows_before_projection",
        "value": len(study)
    },
    {
        "metric": "projected_unique_device_variable_slots",
        "value": len(projected)
    },
    {
        "metric": "devices",
        "value": len(devices)
    },
    {
        "metric": "variables",
        "value": len(variables)
    },
    {
        "metric": "theoretical_slots_per_device",
        "value": len(full_grid)
    },
    {
        "metric": "theoretical_device_time_rows",
        "value": len(devices) * len(full_grid)
    },
    {
        "metric": "theoretical_device_variable_positions",
        "value": (
            len(devices)
            * len(variables)
            * len(full_grid)
        )
    },
    {
        "metric": "available_before_short_gap_interpolation",
        "value": int(
            audit[
                "available_before_interpolation"
            ].sum()
        )
    },
    {
        "metric": "short_gap_values_interpolated",
        "value": int(
            audit[
                "interpolated_short_gap_values"
            ].sum()
        )
    },
    {
        "metric": "available_after_short_gap_interpolation",
        "value": int(
            audit[
                "available_after_interpolation"
            ].sum()
        )
    },
    {
        "metric": "complete_device_time_rows",
        "value": len(complete_segmented)
    },
    {
        "metric": "complete_device_time_pct",
        "value": (
            100
            * len(complete_segmented)
            / len(wide)
        )
    },
    {
        "metric": "contiguous_segments",
        "value": len(segments)
    }
])

dataset_audit.to_csv(
    OUT / "07_dataset_preparation_audit.csv",
    index=False
)

# ============================================================
# 11. PROVENANCE / PROTOCOL MANIFEST
# ============================================================

manifest = pd.DataFrame([
    {
        "parameter": "source_file",
        "value": str(INPUT_FILE)
    },
    {
        "parameter": "source_layer",
        "value": "SPARQL extraction from G_native"
    },
    {
        "parameter": "study_start_utc",
        "value": str(STUDY_START)
    },
    {
        "parameter": "study_end_utc",
        "value": str(STUDY_END)
    },
    {
        "parameter": "grid_frequency",
        "value": FREQ
    },
    {
        "parameter": "slot_projection",
        "value": "nearest 15-minute slot"
    },
    {
        "parameter": "collision_aggregation",
        "value": "arithmetic mean"
    },
    {
        "parameter": "short_gap_interpolation",
        "value": "time interpolation, internal gaps <= 4 slots"
    },
    {
        "parameter": "long_gap_policy",
        "value": "preserved as missing; not interpolated"
    },
    {
        "parameter": "complete_case_rule",
        "value": "temperature, relative_humidity and co2 all available"
    },
    {
        "parameter": "downstream_sequence_rule",
        "value": "ML windows must remain within one contiguous segment"
    }
])

manifest.to_csv(
    OUT / "08_preparation_protocol_manifest.csv",
    index=False
)

# ============================================================
# 12. FINAL VALIDATION
# ============================================================

print("\n====================================")
print("FINAL VALIDATION")
print("====================================")

print(
    dataset_audit.to_string(
        index=False
    )
)

assert len(full_grid) == EXPECTED_GRID_SLOTS
assert len(devices) == EXPECTED_DEVICES
assert set(variables) == EXPECTED_VARIABLES

assert not (
    complete_segmented[
        [
            "temperature",
            "relative_humidity",
            "co2"
        ]
    ]
    .isna()
    .any()
    .any()
)

print("\n✓ Fixed study interval applied")
print("✓ 15-minute projection completed")
print("✓ Collisions aggregated")
print("✓ Only short internal gaps interpolated")
print("✓ Long gaps preserved")
print("✓ Complete T/RH/CO2 records identified")
print("✓ Contiguous segments created")
print("✓ AI-ready synchronized dataset created")

print("\n====================================")
print("FILES CREATED")
print("====================================")

for f in sorted(OUT.iterdir()):
    print(" -", f.name)

